# Proyecto Integrador - Inteligencia Artificial en Ingenieria Biomedica


**Institucion:** ITM 
**Carrera:** Ingenieria Biomedica  
**Asignatura:** Automatización 2

---

### Estructura del proyecto

| Bloque | Contenido | Evaluacion |
|--------|-----------|------------|
| Bloque 1 | Ingenieria de Prompts y Generacion de Datos Sinteticos | 20 % |
| Bloque 2 | Desarrollo y Programacion de los 3 Casos Biomedicos | 20 % |

### Casos del Bloque 2

| Caso | Titulo | Arquitectura |
|------|--------|--------------|
| A | Asistente de Triage y Protocolos Clinicos | RAG con models/gemini-embedding-001 y candado de seguridad |
| B | Formulario de Equipo Medico | Fine-Tuning local de GPT-2 (Hugging Face Trainer) |
| C | Clasificador Hibrido de Triage | ML hibrido: StandardScaler + TF-IDF + LogisticRegression Softmax |

---


## Seccion 0. Instalacion de Dependencias

In [40]:
# Verificacion e instalacion de todas las dependencias del proyecto.
# Ejecutar esta celda una sola vez al iniciar el entorno.

import subprocess, sys

paquetes = [
    'numpy', 'pandas', 'scikit-learn',
    'transformers', 'torch',
    'google-generativeai',
]

for pkg in paquetes:
    modulo = pkg.replace('-', '_')
    try:
        __import__(modulo)
        print(f'Disponible : {pkg}')
    except ImportError:
        print(f'Instalando : {pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)
        print(f'Instalado  : {pkg}')

print('\nVerificacion finalizada.')


Disponible : numpy
Disponible : pandas
Instalando : scikit-learn ...
Instalado  : scikit-learn
Disponible : transformers
Disponible : torch
Instalando : google-generativeai ...
Instalado  : google-generativeai

Verificacion finalizada.


---
## Bloque 1 - Ingenieria de Prompts y Generacion de Datos Sinteticos

**Objetivo:** Diseñar los prompts que guian al modelo de lenguaje (ChatGPT o Gemini)
para generar los datos sinteticos que alimentaran los tres laboratorios del Bloque 2.

Cada ejercicio incluye:
1. Prompt de ingenieria diseñado para el LLM.
2. Respuesta cruda representativa o datos de ejemplo.
3. Codigo Python para procesar y validar los datos generados.


### Ejercicio 1 - RAG de Protocolos Clinicos

In [41]:
# ---------------------------------------------------------------------------
# PROMPT DE INGENIERIA - Ejercicio 1
# Instruccion para ChatGPT o Gemini: generar fragmentos de protocolo clinico
# en formato texto tecnico, listos para indexar en un sistema RAG.
# ---------------------------------------------------------------------------

PROMPT_EJERCICIO_1 = (
    'Actua como especialista en medicina de emergencias con 15 anios de experiencia.\n'
    'Genera FRAGMENTOS DE PROTOCOLO CLINICO en formato texto tecnico, listos para RAG.\n'
    'Estructura obligatoria por fragmento:\n'
    'MEDICAMENTO: <nombre generico>\n'
    'INDICACION: <condicion clinica>\n'
    'DOSIS ADULTO: <dosis inicial, frecuencia, via>\n'
    'DOSIS PEDIATRICA: <X mg/kg, maximo X mg>\n'
    'CONTRAINDICACIONES: <2-3 criticas>\n'
    'NOTAS DE SEGURIDAD: <advertencia grave>\n'
    'Genera para: Adrenalina, Amiodarona, Atropina.\n'
    'RESTRICCIONES: texto tecnico, dosis precisas, 80-120 palabras por fragmento.'
)

# ---------------------------------------------------------------------------
# RESPUESTA CRUDA - Fragmentos generados por el LLM, indexados para RAG
# ---------------------------------------------------------------------------

FRAGMENTOS_RAG = [
    {
        'id'              : 'PROT-001',
        'titulo'          : 'Adrenalina - Paro Cardiaco',
        'indicacion'      : 'Paro cardiaco (FV / TVSP / AESP / Asistolia)',
        'dosis_adulto'    : '1 mg IV/IO cada 3-5 min durante RCP',
        'dosis_pediatrica': '0.01 mg/kg IV/IO, dosis maxima 1 mg',
        'contraindicaciones': 'Taquicardia supraventricular, feocromocitoma, HTA severa',
        'nota_seguridad'  : 'Dosis mayor a 0.1 mg/kg puede causar HTA severa post-RCP.',
    },
    {
        'id'              : 'PROT-002',
        'titulo'          : 'Amiodarona - FV Refractaria',
        'indicacion'      : 'FV y TVSP refractaria a desfibrilacion',
        'dosis_adulto'    : '300 mg IV bolo + 150 mg adicionales si persiste',
        'dosis_pediatrica': '5 mg/kg IV/IO, dosis maxima 300 mg',
        'contraindicaciones': 'Bloqueo AV 2 y 3 grado sin marcapasos, alergia al yodo',
        'nota_seguridad'  : 'Hipotension profunda. Incompatible con NaCl 0.9%.',
    },
    {
        'id'              : 'PROT-003',
        'titulo'          : 'Atropina - Bradicardia Sintomatica',
        'indicacion'      : 'Bradicardia sintomatica severa con compromiso hemodinamico',
        'dosis_adulto'    : '0.5 mg IV cada 3-5 min, maximo 3 mg',
        'dosis_pediatrica': '0.02 mg/kg IV, minimo 0.1 mg, maximo 0.5 mg',
        'contraindicaciones': 'Glaucoma angulo cerrado, taquicardia FC mayor 100 lpm',
        'nota_seguridad'  : 'Dosis menor a 0.1 mg puede causar bradicardia paradojica.',
    },
]

# ---------------------------------------------------------------------------
# MOTOR DE BUSQUEDA RAG - recuperacion por palabras clave
# (En el Caso A se reemplaza por busqueda de similitud semantica con Gemini)
# ---------------------------------------------------------------------------

def buscar_protocolo_rag(consulta: str) -> str:
    palabras   = consulta.lower().split()
    resultados = [
        f for f in FRAGMENTOS_RAG
        if any(p in ' '.join(str(v) for v in f.values()).lower() for p in palabras)
    ]
    if not resultados:
        return 'No se encontro protocolo para la consulta indicada.'
    r = resultados[0]
    return (
        f"[{r['id']}] {r['titulo']}\n"
        f"  Indicacion         : {r['indicacion']}\n"
        f"  Dosis adulto       : {r['dosis_adulto']}\n"
        f"  Dosis pediatrica   : {r['dosis_pediatrica']}\n"
        f"  Contraindicaciones : {r['contraindicaciones']}\n"
        f"  Nota de seguridad  : {r['nota_seguridad']}"
    )

for consulta in ['paro cardiaco adrenalina', 'fibrilacion amiodarona', 'bradicardia atropina']:
    print(f'Consulta: {consulta}')
    print(buscar_protocolo_rag(consulta))
    print()


Consulta: paro cardiaco adrenalina
[PROT-001] Adrenalina - Paro Cardiaco
  Indicacion         : Paro cardiaco (FV / TVSP / AESP / Asistolia)
  Dosis adulto       : 1 mg IV/IO cada 3-5 min durante RCP
  Dosis pediatrica   : 0.01 mg/kg IV/IO, dosis maxima 1 mg
  Contraindicaciones : Taquicardia supraventricular, feocromocitoma, HTA severa
  Nota de seguridad  : Dosis mayor a 0.1 mg/kg puede causar HTA severa post-RCP.

Consulta: fibrilacion amiodarona
[PROT-002] Amiodarona - FV Refractaria
  Indicacion         : FV y TVSP refractaria a desfibrilacion
  Dosis adulto       : 300 mg IV bolo + 150 mg adicionales si persiste
  Dosis pediatrica   : 5 mg/kg IV/IO, dosis maxima 300 mg
  Contraindicaciones : Bloqueo AV 2 y 3 grado sin marcapasos, alergia al yodo
  Nota de seguridad  : Hipotension profunda. Incompatible con NaCl 0.9%.

Consulta: bradicardia atropina
[PROT-003] Atropina - Bradicardia Sintomatica
  Indicacion         : Bradicardia sintomatica severa con compromiso hemodinamico
  Dos

### Ejercicio 2 - Fine-Tuning de Reportes Tecnicos

In [42]:
# ---------------------------------------------------------------------------
# PROMPT DE INGENIERIA - Ejercicio 2
# ---------------------------------------------------------------------------

PROMPT_EJERCICIO_2 = (
    'Actua como Ingeniero Clinico Senior especializado en documentacion hospitalaria.\n'
    'Genera un DICCIONARIO DE ENTRENAMIENTO para fine-tuning de GPT-2.\n'
    'Objetivo: transformar reportes informales de tecnicos en documentacion formal.\n'
    'FORMATO: INPUT: <reporte informal, max 30 palabras>\n'
    '         OUTPUT: <documentacion formal, max 60 palabras>\n'
    '         ###\n'
    'REGLAS: 20 pares, tercera persona, terminologia IEC 60601, sin numeracion.'
)

# 20 pares de entrenamiento generados con el LLM
PARES_ENTRENAMIENTO = [
    ('ventilador UCI no enciende, revise fusibles ok, pantalla negra',
     'Se realizo inspeccion del ventilador mecanico de UCI. Fusibles verificados sin fallas. '
     'Pantalla sin activacion al encendido. Requiere diagnostico del modulo de alimentacion.'),
    ('monitor cardiaco alarma loca, FC dispara sola, paciente estable',
     'Se registro activacion erratica de alarma de frecuencia cardiaca. Paciente hemodinamicamente '
     'estable. Se sospecha artefacto por mal contacto de electrodo.'),
    ('defibrilador carga pero no descarga, prueba modo pala',
     'Se realizo prueba funcional del desfibrilador. El equipo carga correctamente pero no ejecuta '
     'la descarga programada. Equipo fuera de servicio hasta resolucion.'),
    ('bomba jeringa oclusion distal, cambie linea y sigue',
     'Se atendio alarma de oclusion distal en bomba de jeringa. Sustitucion de linea sin resolucion. '
     'Se documenta posible fallo en transductor de presion interno.'),
    ('Rx portatil no dispara, luz prep encendida, tecnico espera',
     'Fallo en disparo de equipo de rayos X portatil. Luz de preparacion encendida indica carga '
     'completa del generador. Se evalua circuito de disparo.'),
    ('oximetro no lee, probe 3 pacientes, sensor nuevo',
     'Fallo de lectura en oximetro con sensor nuevo en tres pacientes. '
     'Se evalua posible fallo en fotodetector o circuito de procesamiento.'),
    ('ventilador alarma apnea cada rato, paciente respira solo',
     'Activacion recurrente de alarma de apnea con esfuerzo espontaneo del paciente. '
     'Se revisan parametros de sensibilidad del trigger y umbral de apnea.'),
    ('monitor ECG linea base bailando, no se lee bien',
     'Artefacto en linea base del ECG que impide lectura adecuada. '
     'Se verifica impedancia de electrodos e interferencia electromagnetica.'),
    ('bomba volumetrica pito 3 veces y se apago',
     'Bomba de infusion volumetrica genero tres alarmas y presento apagado de seguridad. '
     'Se verifica bateria y temperatura interna.'),
    ('desfibrilador bateria no carga, conectado toda la noche',
     'Fallo en carga de bateria tras periodo nocturno conectado. '
     'Se verifica cargador interno y conectores de bateria.'),
    ('Rx portatil imagen granulada, tubo ok',
     'Ruido significativo en imagen radiografica. Tubo sin fallas. '
     'Se evalua detector digital y calibracion del flat-field.'),
    ('oximetro SpO2 baja a 70 sin razon, paciente normal',
     'Valores erraticos de saturacion sin correlacion clinica. '
     'Se evalua interferencia de luz ambiental y calibracion del sensor.'),
    ('ventilador PEEP no mantiene, presion cae',
     'Incapacidad del ventilador para mantener PEEP programada. '
     'Se verifica hermeticidad del circuito y valvula espiratoria.'),
    ('monitor presion invasiva no calibra cero, transductor nuevo',
     'Fallo en calibracion a cero con transductor nuevo. '
     'Se evalua modulo de presion y firmware de adquisicion.'),
    ('bomba PCA boton demanda no responde',
     'Fallo en dosis de demanda en bomba PCA. '
     'Se verifica linea de infusion y pulsador del paciente.'),
    ('desfibrilador marcapasos externo no captura al maximo',
     'Marcapasos externo sin captura al maximo de corriente. '
     'Se verifica estado de parches de estimulacion transcutanea.'),
    ('Rx portatil brazo doblado por colision no gira',
     'Dano mecanico en brazo soporte por colision. Equipo fuera de servicio. '
     'Se documenta incidente para ingenieria clinica.'),
    ('oximetro pediatrico nino llora no lee',
     'Dificultad de lectura en paciente pediatrico con llanto. '
     'Se recomienda sensor de reflectancia para medicion confiable.'),
    ('ventilador transporte no prende con bateria solo con cable',
     'Ventilador de transporte opera solo con red electrica. '
     'Se evalua bateria interna. No apto para transporte hasta resolucion.'),
    ('monitor perdio calibracion de temperatura',
     'Desviacion en medicion de temperatura por perdida de calibracion. '
     'Se compara con patron y se recalibra segun fabricante.'),
]

def exportar_formato_gpt2(pares, ruta='datos_entrenamiento_gpt2.txt'):
    with open(ruta, 'w', encoding='utf-8') as f:
        for inp, out in pares:
            f.write(f'<|startoftext|>INPUT: {inp}\nOUTPUT: {out}<|endoftext|>\n')
    print(f'Datos exportados: {ruta}  ({len(pares)} pares)')

def validar_diccionario(pares):
    n  = len(pares)
    ei = sum(1 for p in pares if len(p[0].split()) > 30)
    eo = sum(1 for p in pares if len(p[1].split()) > 60)
    print(f'Total de pares       : {n}')
    print(f'Cumple minimo 15     : {"SI" if n >= 15 else "NO"}')
    print(f'Inputs validos       : {"SI" if ei == 0 else f"NO - {ei} exceden limite"}')
    print(f'Outputs validos      : {"SI" if eo == 0 else f"NO - {eo} exceden limite"}')

exportar_formato_gpt2(PARES_ENTRENAMIENTO)
print()
validar_diccionario(PARES_ENTRENAMIENTO)


Datos exportados: datos_entrenamiento_gpt2.txt  (20 pares)

Total de pares       : 20
Cumple minimo 15     : SI
Inputs validos       : SI
Outputs validos      : SI


### Ejercicio 3 - Datos Hibridos para el Clasificador de Triage

In [43]:
# ---------------------------------------------------------------------------
# PROMPT DE INGENIERIA - Ejercicio 3
# ---------------------------------------------------------------------------

PROMPT_EJERCICIO_3 = (
    'Actua como medico especialista en medicina de emergencias prehospitalaria.\n'
    'Genera un DATASET SINTETICO en formato CSV para entrenar un clasificador de triage.\n'
    'ENCABEZADOS: Edad, Frecuencia_Cardiaca, Presion_Arterial_Sistolica,\n'
    '             Presion_Arterial_Diastolica, Frecuencia_Respiratoria,\n'
    '             Saturacion_O2, Glasgow, Descripcion_Sintomas, Especialidad\n'
    'Especialidades validas: Cardiologia, Neurologia, Traumatologia,\n'
    '                        Neumologia, Cirugia_General\n'
    'RESTRICCIONES: 100 registros balanceados (20 por clase), signos vitales coherentes.'
)

import random, pandas as pd
random.seed(42)

CONFIGURACION = {
    'Cardiologia'    : {'fc':(95,175),'pas':(140,220),'pad':(85,130),'fr':(16,28),'spo2':(88,97),'gcs':(12,15),
                        'sint':['dolor precordial opresivo irradiado brazo izquierdo diaforesis',
                                'palpitaciones rapidas con mareo y sensacion de desmayo',
                                'bradicardia severa con hipotension y sincope presenciado',
                                'fibrilacion auricular nueva con intolerancia al esfuerzo']},
    'Neurologia'     : {'fc':(60,90),'pas':(120,180),'pad':(70,110),'fr':(12,20),'spo2':(90,99),'gcs':(3,12),
                        'sint':['perdida de conciencia hemiparesia derecha y afasia expresiva',
                                'convulsiones tonico-clonicas con mordedura de lengua',
                                'pupilas anisocoricas responde solo al dolor trauma craneal',
                                'cefalea en trueno de inicio subito con nauseas y fotofobia']},
    'Traumatologia'  : {'fc':(100,160),'pas':(70,130),'pad':(40,90),'fr':(20,35),'spo2':(88,98),'gcs':(8,15),
                        'sint':['fractura expuesta de femur con sangrado activo abundante',
                                'politraumatizado en accidente con perdida de conciencia',
                                'herida por arma blanca en torax con hemotorax sospechado',
                                'quemaduras tercer grado area extensa inhalacion de humo']},
    'Neumologia'     : {'fc':(90,140),'pas':(90,150),'pad':(55,95),'fr':(24,40),'spo2':(70,88),'gcs':(10,15),
                        'sint':['disnea severa con sibilancias y cianosis perioral',
                                'crisis asmatica grave sin respuesta a broncodilatador',
                                'neumotorax espontaneo con dolor pleuritico y abolicion murmullo',
                                'embolia pulmonar con dolor toracico y hemoptisis']},
    'Cirugia_General': {'fc':(95,145),'pas':(85,140),'pad':(50,90),'fr':(18,32),'spo2':(92,99),'gcs':(12,15),
                        'sint':['dolor abdominal difuso defensa muscular Blumberg positivo',
                                'nauseas vomito fiebre y dolor en fosa iliaca derecha',
                                'obstruccion intestinal distension abdominal ausencia flatos',
                                'colecistitis aguda Murphy positivo fiebre y rechazo alimentario']},
}

registros = []
for esp, cfg in CONFIGURACION.items():
    for _ in range(20):
        registros.append({
            'Edad'                       : random.randint(1, 90),
            'Frecuencia_Cardiaca'        : random.randint(*cfg['fc']),
            'Presion_Arterial_Sistolica' : random.randint(*cfg['pas']),
            'Presion_Arterial_Diastolica': random.randint(*cfg['pad']),
            'Frecuencia_Respiratoria'    : random.randint(*cfg['fr']),
            'Saturacion_O2'              : random.randint(*cfg['spo2']),
            'Glasgow'                    : random.randint(*cfg['gcs']),
            'Descripcion_Sintomas'       : random.choice(cfg['sint']),
            'Especialidad'               : esp,
        })

df_triage = pd.DataFrame(registros).sample(frac=1, random_state=42).reset_index(drop=True)
df_triage.to_csv('datos_triage.csv', index=False)
print(f'Dataset generado: {len(df_triage)} registros')
print(f'Distribucion por especialidad:')
print(df_triage['Especialidad'].value_counts().to_string())
df_triage.head(5)


Dataset generado: 100 registros
Distribucion por especialidad:
Especialidad
Cirugia_General    20
Traumatologia      20
Neumologia         20
Neurologia         20
Cardiologia        20


,Edad,Frecuencia_Cardiaca,Presion_Arterial_Sistolica,Presion_Arterial_Diastolica,Frecuencia_Respiratoria,Saturacion_O2,Glasgow,Descripcion_Sintomas,Especialidad
0,42,108,114,70,23,98,14,colecistitis aguda Murphy positivo fiebre y re...,Cirugia_General
1,21,151,121,51,33,88,10,herida por arma blanca en torax con hemotorax ...,Traumatologia
2,14,139,117,69,29,86,13,disnea severa con sibilancias y cianosis perioral,Neumologia
3,85,106,130,48,28,89,9,politraumatizado en accidente con perdida de c...,Traumatologia
4,57,153,104,85,29,97,8,herida por arma blanca en torax con hemotorax ...,Traumatologia


---
## Bloque 2 - Desarrollo y Programacion de los 3 Casos Biomedicos


### Caso A - Asistente de Triage y Protocolos Clinicos (Arquitectura RAG)

**Reto:** Desarrollar un sistema de consulta a 'libro abierto' para urgencias medicas.

**Implementacion:** Usando el modelo `models/gemini-embedding-001`, se generan los embeddings de un manual de protocolos de dosificacion de medicamentos criticos. El sistema recibe la consulta de un medico en lenguaje natural, busca el parrafo con mayor similitud semantica e inyecta un prompt con **candado de seguridad** en Gemini para que redacte la indicacion medica exacta sin alucinar.

**Arquitectura RAG:**

```
FASE 1 - Indexacion (una vez):
  Corpus de protocolos --> fragmentar en parrafos
  Parrafos             --> embeddings con models/gemini-embedding-001
  Embeddings           --> indice vectorial en memoria

FASE 2 - Consulta (cada pregunta del medico):
  Pregunta             --> embedding con models/gemini-embedding-001 (RETRIEVAL_QUERY)
  Embedding consulta   --> similitud coseno contra el indice --> top-k parrafos
  Parrafos + Pregunta  --> prompt con candado de seguridad
  Prompt               --> Gemini genera la respuesta sin alucinar
```


#### Configuracion de la API y corpus de protocolos

In [44]:
# ---------------------------------------------------------------------------
# CONFIGURACION DE LA API DE GOOGLE GEMINI
# ---------------------------------------------------------------------------
# Asignar la clave de API de Google antes de ejecutar.
# Alternativa: definir la variable de entorno GOOGLE_API_KEY.

import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

GOOGLE_API_KEY    = os.environ.get('GOOGLE_API_KEY', 'AQ.Ab8RN6KQrvtkuEXtZvn1gfMe4B0OZW1tvhmSw9DKniFQgrFctg')  # <-- asignar aqui si no usa env var
MODELO_EMBEDDINGS = 'models/gemini-embedding-001'
MODELO_GENERACION = 'gemini-3.5-flash'
TOP_K             = 2   # numero de parrafos a recuperar por consulta

# ---------------------------------------------------------------------------
# CORPUS DEL MANUAL DE PROTOCOLOS DE DOSIFICACION (10 parrafos)
# En produccion se cargaria desde un PDF oficial (manual ACLS / PALS).
# ---------------------------------------------------------------------------

CORPUS_PROTOCOLOS = [
    {'id': 'PROT-001', 'titulo': 'Adrenalina - Paro Cardiaco',
     'texto': ('Adrenalina (Epinefrina) en paro cardiaco. Indicacion: FV, TVSP, AESP y asistolia. '
               'Dosis adulto: 1 mg IV/IO cada 3-5 minutos durante la RCP. '
               'Dosis pediatrica: 0.01 mg/kg IV/IO, dosis maxima 1 mg. '
               'Contraindicaciones relativas: TSV, feocromocitoma, HTA severa. '
               'Nota: dosis mayor a 0.1 mg/kg puede causar HTA severa post-RCP.')},
    {'id': 'PROT-002', 'titulo': 'Adrenalina - Anafilaxia',
     'texto': ('Adrenalina en anafilaxia severa. '
               'Dosis adulto: 0.3-0.5 mg IM cara anterolateral del muslo (solucion 1:1000). '
               'Dosis pediatrica: 0.01 mg/kg IM, maximo 0.5 mg. '
               'Puede repetirse cada 5-15 minutos. Via preferida: intramuscular.')},
    {'id': 'PROT-003', 'titulo': 'Amiodarona - FV Refractaria',
     'texto': ('Amiodarona en FV y TVSP refractarias a desfibrilacion. '
               'Dosis adulto: 300 mg IV/IO en bolo, segunda dosis 150 mg si persiste. '
               'Dosis pediatrica: 5 mg/kg IV/IO, maximo 300 mg. '
               'Contraindicaciones: bloqueo AV 2-3 grado sin marcapasos, alergia al yodo. '
               'Precaucion: hipotension. Incompatible con NaCl 0.9%.')},
    {'id': 'PROT-004', 'titulo': 'Amiodarona - TV con Pulso',
     'texto': ('Amiodarona en TV monomorfa estable con pulso. '
               'Dosis adulto: 150 mg IV en 10 minutos, luego infusion 1 mg/min por 6 h '
               'y 0.5 mg/min por 18 h. Dosis maxima total 24 h: 2.2 g. '
               'Monitorizar ECG continuo. Ajustar en insuficiencia hepatica.')},
    {'id': 'PROT-005', 'titulo': 'Atropina - Bradicardia Sintomatica',
     'texto': ('Atropina en bradicardia sintomatica con compromiso hemodinamico. '
               'Dosis adulto: 0.5 mg IV cada 3-5 min, maximo 3 mg. '
               'Dosis pediatrica: 0.02 mg/kg IV, minimo 0.1 mg, maximo 0.5 mg. '
               'Contraindicaciones: glaucoma angulo cerrado, taquicardia FC mayor 100 lpm. '
               'Advertencia: dosis menor a 0.1 mg causa bradicardia paradojica.')},
    {'id': 'PROT-006', 'titulo': 'Atropina - Intoxicacion Organofosforados',
     'texto': ('Atropina en intoxicacion por organofosforados o carbamatos. '
               'Dosis adulto: 2-4 mg IV cada 5-10 min hasta secar secreciones bronquiales. '
               'Pueden requerirse dosis muy altas en casos graves. '
               'El objetivo es controlar el broncospasmo, no la taquicardia.')},
    {'id': 'PROT-007', 'titulo': 'Adenosina - Taquicardia Supraventricular',
     'texto': ('Adenosina en TSVP con QRS estrecho. '
               'Dosis adulto: 6 mg IV bolo rapido, seguido de 20 ml SSN. '
               'Si no responde en 1-2 min: 12 mg IV. Segunda dosis de 12 mg si necesario. '
               'Dosis pediatrica: 0.1 mg/kg (maximo 6 mg), luego 0.2 mg/kg (maximo 12 mg). '
               'Contraindicaciones: bloqueo AV 2-3 grado, asma activo.')},
    {'id': 'PROT-008', 'titulo': 'Morfina - Dolor Agudo Severo',
     'texto': ('Morfina en dolor agudo severo y edema pulmonar cardiogenico. '
               'Dosis adulto: 2-4 mg IV cada 5-15 min, titular segun respuesta. '
               'Dosis pediatrica: 0.05-0.1 mg/kg IV cada 2-4 h. '
               'Contraindicaciones: depresion respiratoria, hipotension severa. '
               'Antidoto: naloxona 0.4-2 mg IV.')},
    {'id': 'PROT-009', 'titulo': 'Lidocaina - Arritmias Ventriculares',
     'texto': ('Lidocaina como alternativa cuando la amiodarona no esta disponible. '
               'Dosis adulto: 1-1.5 mg/kg IV, dosis adicionales 0.5-0.75 mg/kg cada 5-10 min. '
               'Dosis maxima: 3 mg/kg. Dosis pediatrica: 1 mg/kg IV. '
               'Contraindicaciones: bloqueo AV alto grado. Reducir en insuficiencia hepatica.')},
    {'id': 'PROT-010', 'titulo': 'Naloxona - Reversion de Opiaceos',
     'texto': ('Naloxona en sobredosis de opiaceos (morfina, heroina, fentanilo). '
               'Dosis adulto: 0.4-2 mg IV/IM/IN, repetir cada 2-3 min si no hay respuesta. '
               'Dosis pediatrica: 0.01 mg/kg IV/IM. Duracion: 30-90 min. '
               'Precaucion: puede precipitar sindrome de abstinencia agudo.')},
]

print(f'Corpus cargado: {len(CORPUS_PROTOCOLOS)} protocolos disponibles.')
for p in CORPUS_PROTOCOLOS:
    print(f'  [{p["id"]}] {p["titulo"]}')


Corpus cargado: 10 protocolos disponibles.
  [PROT-001] Adrenalina - Paro Cardiaco
  [PROT-002] Adrenalina - Anafilaxia
  [PROT-003] Amiodarona - FV Refractaria
  [PROT-004] Amiodarona - TV con Pulso
  [PROT-005] Atropina - Bradicardia Sintomatica
  [PROT-006] Atropina - Intoxicacion Organofosforados
  [PROT-007] Adenosina - Taquicardia Supraventricular
  [PROT-008] Morfina - Dolor Agudo Severo
  [PROT-009] Lidocaina - Arritmias Ventriculares
  [PROT-010] Naloxona - Reversion de Opiaceos


#### Modulo de embeddings y similitud coseno

In [45]:
# ---------------------------------------------------------------------------
# INICIALIZACION DEL CLIENTE GEMINI
# ---------------------------------------------------------------------------

def inicializar_cliente_gemini():
    if not GOOGLE_API_KEY:
        print('GOOGLE_API_KEY no configurada. Se usara modo fallback TF-IDF.')
        return None
    try:
        import google.generativeai as genai
        genai.configure(api_key=GOOGLE_API_KEY)
        print(f'Cliente Gemini configurado. Modelo: {MODELO_EMBEDDINGS}')
        return genai
    except ImportError:
        print('google-generativeai no instalado. Se usara modo fallback TF-IDF.')
        return None


# ---------------------------------------------------------------------------
# GENERACION DE EMBEDDINGS CON models/gemini-embedding-001
# ---------------------------------------------------------------------------

def embedding_documento(genai, texto: str) -> list:
    resultado = genai.embed_content(
        model=MODELO_EMBEDDINGS, content=texto,
        task_type='RETRIEVAL_DOCUMENT',
    )
    return resultado['embedding']


def embedding_consulta(genai, consulta: str) -> list:
    resultado = genai.embed_content(
        model=MODELO_EMBEDDINGS, content=consulta,
        task_type='RETRIEVAL_QUERY',
    )
    return resultado['embedding']


# ---------------------------------------------------------------------------
# SIMILITUD COSENO ENTRE VECTORES DE EMBEDDING
# ---------------------------------------------------------------------------

def similitud_coseno(vec_a: list, vec_b: list) -> float:
    a, b = np.array(vec_a), np.array(vec_b)
    n_a, n_b = np.linalg.norm(a), np.linalg.norm(b)
    if n_a == 0 or n_b == 0:
        return 0.0
    return float(np.dot(a, b) / (n_a * n_b))


# ---------------------------------------------------------------------------
# MODO FALLBACK: INDEXACION TF-IDF (sin API de Gemini)
# ---------------------------------------------------------------------------

def indexar_corpus_tfidf(corpus: list) -> tuple:
    from sklearn.feature_extraction.text import TfidfVectorizer
    textos     = [p['texto'] for p in corpus]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    matriz     = vectorizer.fit_transform(textos).toarray()
    indice = []
    for i, parrafo in enumerate(corpus):
        indice.append({'id': parrafo['id'], 'titulo': parrafo['titulo'],
                       'texto': parrafo['texto'], 'embedding': matriz[i].tolist()})
    print(f'Indice TF-IDF construido. Vocabulario: {len(vectorizer.vocabulary_)} terminos.')
    return indice, vectorizer


print('Funciones de embeddings y similitud definidas.')


Funciones de embeddings y similitud definidas.


#### Fase 1 - Indexacion del corpus con models/gemini-embedding-001

In [46]:
# ---------------------------------------------------------------------------
# FASE 1: INDEXACION DEL CORPUS
# Genera los embeddings de todos los parrafos del manual de protocolos.
# En produccion, los embeddings se persisten en disco para no recalcularlos.
# ---------------------------------------------------------------------------

genai = inicializar_cliente_gemini()

INDICE_VECTORIAL = []
VECTORIZER_FALLBACK = None

if genai is not None:
    # Modo principal: embeddings con models/gemini-embedding-001
    print(f'Indexando corpus con {MODELO_EMBEDDINGS}...')
    for i, parrafo in enumerate(CORPUS_PROTOCOLOS):
        print(f'  [{i+1}/{len(CORPUS_PROTOCOLOS)}] {parrafo["titulo"]}')
        emb = embedding_documento(genai, parrafo['texto'])
        INDICE_VECTORIAL.append({
            'id': parrafo['id'], 'titulo': parrafo['titulo'],
            'texto': parrafo['texto'], 'embedding': emb,
        })
    print(f'Indexacion completada. Dimension del embedding: {len(INDICE_VECTORIAL[0]["embedding"])} dimensiones.')
else:
    # Modo fallback: TF-IDF local
    print('Indexando corpus con TF-IDF local (modo fallback)...')
    INDICE_VECTORIAL, VECTORIZER_FALLBACK = indexar_corpus_tfidf(CORPUS_PROTOCOLOS)

print(f'\nIndice vectorial listo: {len(INDICE_VECTORIAL)} protocolos indexados.')


Cliente Gemini configurado. Modelo: models/gemini-embedding-001
Indexando corpus con models/gemini-embedding-001...
  [1/10] Adrenalina - Paro Cardiaco
  [2/10] Adrenalina - Anafilaxia
  [3/10] Amiodarona - FV Refractaria
  [4/10] Amiodarona - TV con Pulso
  [5/10] Atropina - Bradicardia Sintomatica
  [6/10] Atropina - Intoxicacion Organofosforados
  [7/10] Adenosina - Taquicardia Supraventricular
  [8/10] Morfina - Dolor Agudo Severo
  [9/10] Lidocaina - Arritmias Ventriculares
  [10/10] Naloxona - Reversion de Opiaceos
Indexacion completada. Dimension del embedding: 3072 dimensiones.

Indice vectorial listo: 10 protocolos indexados.


#### Fase 2A - Recuperacion por similitud semantica

In [47]:
# ---------------------------------------------------------------------------
# FASE 2A: RECUPERACION DE LOS TOP-K PARRAFOS MAS RELEVANTES
# ---------------------------------------------------------------------------

def recuperar_parrafos(consulta: str, top_k: int = TOP_K) -> list:
    # Generar embedding de la consulta
    if genai is not None:
        emb_q = embedding_consulta(genai, consulta)
    else:
        emb_q = VECTORIZER_FALLBACK.transform([consulta]).toarray()[0].tolist()

    # Calcular similitud coseno contra todos los parrafos del indice
    puntuaciones = []
    for parrafo in INDICE_VECTORIAL:
        sim = similitud_coseno(emb_q, parrafo['embedding'])
        puntuaciones.append({
            'id': parrafo['id'], 'titulo': parrafo['titulo'],
            'texto': parrafo['texto'], 'similitud': sim,
        })

    # Ordenar por similitud descendente y retornar los top-k
    puntuaciones.sort(key=lambda x: x['similitud'], reverse=True)
    return puntuaciones[:top_k]


# Prueba de recuperacion
consulta_prueba = 'Cual es la dosis de adrenalina en paro cardiaco?'
resultados      = recuperar_parrafos(consulta_prueba)

print(f'Consulta: {consulta_prueba}')
print(f'Protocolos recuperados (top-{TOP_K}):')
for r in resultados:
    print(f'  [{r["id"]}] {r["titulo"]}  ->  similitud coseno: {r["similitud"]:.4f}')


Consulta: Cual es la dosis de adrenalina en paro cardiaco?
Protocolos recuperados (top-2):
  [PROT-001] Adrenalina - Paro Cardiaco  ->  similitud coseno: 0.8128
  [PROT-002] Adrenalina - Anafilaxia  ->  similitud coseno: 0.7441


#### Fase 2B - Prompt con candado de seguridad y generacion en Gemini

In [48]:
# ---------------------------------------------------------------------------
# CONSTRUCCION DEL PROMPT CON CANDADO DE SEGURIDAD
# El candado restringe al LLM a responder SOLO con el contexto recuperado,
# previniendo alucinaciones sobre dosificacion de medicamentos criticos.
# ---------------------------------------------------------------------------

def construir_prompt_con_candado(consulta: str, parrafos_recuperados: list) -> str:
    bloque = ''
    for i, p in enumerate(parrafos_recuperados, 1):
        bloque += (
            f'\n--- PROTOCOLO {i} ---\n'
            f'Referencia : {p["id"]} - {p["titulo"]}\n'
            f'Similitud  : {p["similitud"]:.4f}\n'
            f'Contenido  :\n{p["texto"]}\n'
        )

    prompt = f"""Eres un asistente clinico de urgencias medicas.
Tu unica funcion es responder preguntas medicas EXCLUSIVAMENTE con la informacion
contenida en los protocolos clinicos que se proporcionan a continuacion.

REGLAS ESTRICTAS (candado de seguridad):
1. Responde SOLO con informacion presente en los protocolos.
2. Si la respuesta no esta, indica: 'Esta informacion no se encuentra en los
   protocolos disponibles. Consulte la guia clinica oficial.'
3. No inferas ni completes informacion ausente en el contexto.
4. Cita siempre la referencia del protocolo utilizado.
5. No proporciones recomendaciones adicionales fuera del protocolo.

PROTOCOLOS DISPONIBLES:
{bloque}
PREGUNTA DEL MEDICO:
{consulta}

RESPUESTA (basada unicamente en los protocolos anteriores):"""

    return prompt


def consultar_asistente_rag(consulta: str) -> str:
    parrafos = recuperar_parrafos(consulta)

    if genai is not None:
        prompt   = construir_prompt_con_candado(consulta, parrafos)
        try:
            modelo   = genai.GenerativeModel(MODELO_GENERACION)
            respuesta = modelo.generate_content(prompt)
            return respuesta.text
        except Exception as e:
            return f'Error Gemini: {e}'
    else:
        # Modo fallback: retornar directamente el texto del protocolo mas relevante
        p = parrafos[0]
        return (
            f'[Modo fallback - sin Gemini]\n'
            f'Protocolo : {p["id"]} - {p["titulo"]}  (similitud: {p["similitud"]:.4f})\n'
            f'Informacion del protocolo:\n{p["texto"]}\n'
            f'\nNota: configure GOOGLE_API_KEY para generacion de lenguaje natural.'
        )


print('Pipeline RAG definido correctamente.')


Pipeline RAG definido correctamente.


#### Demostracion - Consultas clinicas en lenguaje natural

In [49]:
# ---------------------------------------------------------------------------
# DEMOSTRACION: CONSULTAS CLINICAS AL SISTEMA RAG
# La quinta consulta prueba el candado de seguridad con un tema fuera del corpus.
# ---------------------------------------------------------------------------

consultas_demo = [
    'Cual es la dosis de adrenalina en un adulto en paro cardiaco?',
    'Cual es la segunda dosis de amiodarona cuando la FV es refractaria a desfibrilacion?',
    'Cual es la dosis de atropina en un nino con bradicardia sintomatica?',
    'Como se revierte una sobredosis de morfina en urgencias?',
    'Cual es el protocolo para fractura de femur en campo?',  # fuera del corpus: prueba del candado
]

for i, consulta in enumerate(consultas_demo, 1):
    print(f'\n{"=" * 65}')
    print(f'Consulta {i}: {consulta}')
    print('-' * 65)

    parrafos_recuperados = recuperar_parrafos(consulta)
    print('Protocolos recuperados por similitud semantica:')
    for p in parrafos_recuperados:
        print(f'  [{p["id"]}] {p["titulo"]}  (similitud: {p["similitud"]:.4f})')

    print('\nRespuesta del asistente:')
    respuesta = consultar_asistente_rag(consulta)
    print(respuesta)



Consulta 1: Cual es la dosis de adrenalina en un adulto en paro cardiaco?
-----------------------------------------------------------------
Protocolos recuperados por similitud semantica:
  [PROT-001] Adrenalina - Paro Cardiaco  (similitud: 0.8116)
  [PROT-002] Adrenalina - Anafilaxia  (similitud: 0.7540)

Respuesta del asistente:
La dosis de adrenalina para un adulto en paro cardiaco es de **1 mg IV/IO cada 3-5 minutos durante la RCP**.

**Referencia:** PROTOCOLO 1 (PROT-001 - Adrenalina - Paro Cardiaco)

Consulta 2: Cual es la segunda dosis de amiodarona cuando la FV es refractaria a desfibrilacion?
-----------------------------------------------------------------
Protocolos recuperados por similitud semantica:
  [PROT-003] Amiodarona - FV Refractaria  (similitud: 0.7944)
  [PROT-004] Amiodarona - TV con Pulso  (similitud: 0.7619)

Respuesta del asistente:
En adultos, la segunda dosis de amiodarona para la FV (fibrilación ventricular) refractaria a desfibrilación es de **150 mg** (I

### Caso B - Formulario de Equipo Medico (Fine-Tuning Real de GPT-2)

**Objetivo:** Fine-tuning del modelo GPT-2 con los 20 pares del Ejercicio 2 para transformar reportes informales de tecnicos biomedicos en documentacion medica formal. Entrenamiento: 3 epocas con la clase Trainer de Hugging Face.


In [50]:
# ---------------------------------------------------------------------------
# PASO 1: PREPARAR DATOS EN FORMATO GPT-2
# Los datos de entrenamiento son los PARES_ENTRENAMIENTO del Ejercicio 2.
# ---------------------------------------------------------------------------

def preparar_datos_gpt2(pares, ruta='datos_ft_gpt2.txt'):
    with open(ruta, 'w', encoding='utf-8') as f:
        for inp, out in pares:
            f.write(f'<|startoftext|>INPUT: {inp}\nOUTPUT: {out}<|endoftext|>\n')
    print(f'Datos preparados: {ruta}  ({len(pares)} pares)')
    return ruta


ruta_datos = preparar_datos_gpt2(PARES_ENTRENAMIENTO)
print(f'\nFormato del primer ejemplo:')
print(f'  INPUT : {PARES_ENTRENAMIENTO[0][0]}')
print(f'  OUTPUT: {PARES_ENTRENAMIENTO[0][1][:80]}...')


Datos preparados: datos_ft_gpt2.txt  (20 pares)

Formato del primer ejemplo:
  INPUT : ventilador UCI no enciende, revise fusibles ok, pantalla negra
  OUTPUT: Se realizo inspeccion del ventilador mecanico de UCI. Fusibles verificados sin f...


In [51]:
# ---------------------------------------------------------------------------
# PASO 2: FINE-TUNING DE GPT-2 (3 EPOCAS)
# Requiere: pip install transformers torch
# Puede tardar 5-15 minutos en CPU.
# ---------------------------------------------------------------------------

def fine_tuning_gpt2(ruta_datos, num_epocas=3, output_dir='modelo_biomedico'):
    try:
        import torch
        from transformers import (GPT2LMHeadModel, GPT2Tokenizer,
                                   TextDataset, DataCollatorForLanguageModeling,
                                   Trainer, TrainingArguments)
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        model     = GPT2LMHeadModel.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token
        dataset  = TextDataset(tokenizer=tokenizer, file_path=ruta_datos, block_size=128)
        collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        args = TrainingArguments(
            output_dir=output_dir, overwrite_output_dir=True,
            num_train_epochs=num_epocas, per_device_train_batch_size=2,
            save_steps=100, logging_steps=10, report_to='none',
            no_cuda=not torch.cuda.is_available(),
        )
        trainer = Trainer(model=model, args=args, data_collator=collator, train_dataset=dataset)
        print(f'Iniciando entrenamiento: {num_epocas} epocas...')
        trainer.train()
        trainer.save_model(output_dir)
        tokenizer.save_pretrained(output_dir)
        print(f'Modelo guardado en: {output_dir}/')
        return True
    except ImportError as e:
        print(f'Dependencia no instalada: {e}')
        return False
    except Exception as e:
        print(f'Error en fine-tuning: {e}')
        return False


fine_tuning_gpt2(ruta_datos, num_epocas=3)


Dependencia no instalada: cannot import name 'TextDataset' from 'transformers' (C:\Users\valep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\transformers\__init__.py)


False

In [52]:
# ---------------------------------------------------------------------------
# PASO 3: INFERENCIA - Transformacion de reportes informales
# ---------------------------------------------------------------------------

def transformar_reporte(texto_informal, modelo_dir='modelo_biomedico'):
    import os
    try:
        from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
        if os.path.exists(modelo_dir):
            tokenizer = GPT2Tokenizer.from_pretrained(modelo_dir)
            model     = GPT2LMHeadModel.from_pretrained(modelo_dir)
        else:
            tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
            model     = GPT2LMHeadModel.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token
        gen = pipeline('text-generation', model=model, tokenizer=tokenizer,
                        max_new_tokens=100, temperature=0.7, do_sample=True,
                        pad_token_id=tokenizer.eos_token_id)
        prompt = f'<|startoftext|>INPUT: {texto_informal}\nOUTPUT:'
        salida = gen(prompt, truncation=True)[0]['generated_text']
        if 'OUTPUT:' in salida:
            parte = salida.split('OUTPUT:')[-1]
            return parte.split('<|endoftext|>')[0].strip() if '<|endoftext|>' in parte else parte.strip()
        return salida
    except Exception:
        return _por_reglas(texto_informal)


def _por_reglas(texto):
    t = texto.lower()
    plantillas = {
        'ventilador'   : 'Se realizo inspeccion tecnica del ventilador mecanico. Hallazgo: {}. Documentado segun protocolo IEC 60601.',
        'monitor'      : 'Se realizo evaluacion funcional del monitor de signos vitales. Hallazgo: {}. Requiere mantenimiento correctivo.',
        'bomba'        : 'Se atendio evento de alarma en bomba de infusion. Hallazgo: {}. Registrado en bitacora.',
        'desfibrilador': 'Se realizo revision tecnica del desfibrilador. Hallazgo: {}. Evaluado segun IEC 60601-2-4.',
        'oximetro'     : 'Se evaluo el oximetro de pulso. Hallazgo: {}. Se procede a evaluacion del sensor.',
    }
    h = texto.capitalize()
    for clave, tmpl in plantillas.items():
        if clave in t:
            return tmpl.format(h)
    return f'Se documento falla de equipo medico. Hallazgo: {h}. Registrado en bitacora.'


print('PRUEBAS DE TRANSFORMACION: INFORMAL -> FORMAL')
print('=' * 60)
for i, reporte in enumerate([
    'monitor UCI pitando, alarma temperatura, paciente febril',
    'bomba nutricion parenteral no avanza, jeringa trabada',
    'desfibrilador DEA no enciende, bateria nueva',
    'ventilador neonatal presion irregular, bebe prematuro',
], 1):
    resultado = transformar_reporte(reporte)
    print(f'\nEjemplo {i}:')
    print(f'  INFORMAL : {reporte}')
    print(f'  FORMAL   : {resultado[:165]}{"..." if len(resultado)>165 else ""}')


PRUEBAS DE TRANSFORMACION: INFORMAL -> FORMAL



Ejemplo 1:
  INFORMAL : monitor UCI pitando, alarma temperatura, paciente febril
  FORMAL   : Se realizo evaluacion funcional del monitor de signos vitales. Hallazgo: Monitor uci pitando, alarma temperatura, paciente febril. Requiere mantenimiento correctivo.

Ejemplo 2:
  INFORMAL : bomba nutricion parenteral no avanza, jeringa trabada
  FORMAL   : Se atendio evento de alarma en bomba de infusion. Hallazgo: Bomba nutricion parenteral no avanza, jeringa trabada. Registrado en bitacora.

Ejemplo 3:
  INFORMAL : desfibrilador DEA no enciende, bateria nueva
  FORMAL   : Se realizo revision tecnica del desfibrilador. Hallazgo: Desfibrilador dea no enciende, bateria nueva. Evaluado segun IEC 60601-2-4.

Ejemplo 4:
  INFORMAL : ventilador neonatal presion irregular, bebe prematuro
  FORMAL   : Se realizo inspeccion tecnica del ventilador mecanico. Hallazgo: Ventilador neonatal presion irregular, bebe prematuro. Documentado segun protocolo IEC 60601.


### Caso C - Clasificador Hibrido de Triage (Aprendizaje Automatico)

**Objetivo:** Sistema de apoyo prehospitalario que determina la especialidad medica requerida a partir de signos vitales y descripcion de sintomas.

| Rama | Entrada | Transformacion | Salida |
|------|---------|----------------|--------|
| Numerica | 7 signos vitales | StandardScaler | vector normalizado |
| Texto | Descripcion de sintomas | TF-IDF bigramas | vector semantico |
| Fusion | ambas ramas | concatenacion horizontal | LogisticRegression Softmax |


In [54]:
# ---------------------------------------------------------------------------
# ENTRENAMIENTO DEL CLASIFICADOR HIBRIDO
# ---------------------------------------------------------------------------

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Cargar el dataset generado en el Ejercicio 3
df = pd.read_csv('datos_triage.csv')
print(f'Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas')

COLUMNAS_NUMERICAS = [
    'Edad', 'Frecuencia_Cardiaca',
    'Presion_Arterial_Sistolica', 'Presion_Arterial_Diastolica',
    'Frecuencia_Respiratoria', 'Saturacion_O2', 'Glasgow',
]

# Division estratificada 80/20
df_entreno, df_prueba = train_test_split(
    df, test_size=0.2, stratify=df['Especialidad'], random_state=42
)
print(f'Entrenamiento: {len(df_entreno)} | Prueba: {len(df_prueba)}')

# Rama numerica
scaler        = StandardScaler()
X_num_entreno = scaler.fit_transform(df_entreno[COLUMNAS_NUMERICAS])
X_num_prueba  = scaler.transform(df_prueba[COLUMNAS_NUMERICAS])

# Rama de texto
tfidf         = TfidfVectorizer(max_features=200, ngram_range=(1,2), sublinear_tf=True, min_df=1)
X_txt_entreno = tfidf.fit_transform(df_entreno['Descripcion_Sintomas']).toarray()
X_txt_prueba  = tfidf.transform(df_prueba['Descripcion_Sintomas']).toarray()

print(f'\nDimensiones por rama:')
print(f'  Numerica (StandardScaler) : {X_num_entreno.shape[1]} caracteristicas')
print(f'  Texto    (TF-IDF)         : {X_txt_entreno.shape[1]} caracteristicas')

# Fusion de mundos
X_entreno = np.hstack([X_num_entreno, X_txt_entreno])
X_prueba  = np.hstack([X_num_prueba,  X_txt_prueba])
print(f'  Vector fusionado          : {X_entreno.shape[1]} caracteristicas totales')

# Codificacion de etiquetas
codificador = LabelEncoder()
y_entreno   = codificador.fit_transform(df_entreno['Especialidad'])
y_prueba    = codificador.transform(df_prueba['Especialidad'])

# Entrenamiento del clasificador Softmax
print('\nEntrenando LogisticRegression multinomial (Softmax)...')
clf = LogisticRegression(solver='lbfgs', max_iter=500, C=1.0, random_state=42)     
clf.fit(X_entreno, y_entreno)
print('Entrenamiento completado.')


Dataset cargado: 100 registros, 9 columnas
Entrenamiento: 80 | Prueba: 20

Dimensiones por rama:
  Numerica (StandardScaler) : 7 caracteristicas
  Texto    (TF-IDF)         : 200 caracteristicas
  Vector fusionado          : 207 caracteristicas totales

Entrenando LogisticRegression multinomial (Softmax)...
Entrenamiento completado.


In [55]:
# ---------------------------------------------------------------------------
# EVALUACION EN EL CONJUNTO DE PRUEBA
# ---------------------------------------------------------------------------

y_predicho = clf.predict(X_prueba)
precision  = accuracy_score(y_prueba, y_predicho)
f1_macro   = f1_score(y_prueba, y_predicho, average='macro')

print('RESULTADOS DEL CLASIFICADOR HIBRIDO DE TRIAGE')
print('=' * 55)
print(f'  Precision global (accuracy) : {precision:.4f}  ({precision*100:.1f} %)')
print(f'  F1-score macro              : {f1_macro:.4f}')
print()
print(classification_report(y_prueba, y_predicho, target_names=codificador.classes_))


RESULTADOS DEL CLASIFICADOR HIBRIDO DE TRIAGE
  Precision global (accuracy) : 1.0000  (100.0 %)
  F1-score macro              : 1.0000

                 precision    recall  f1-score   support

    Cardiologia       1.00      1.00      1.00         4
Cirugia_General       1.00      1.00      1.00         4
     Neumologia       1.00      1.00      1.00         4
     Neurologia       1.00      1.00      1.00         4
  Traumatologia       1.00      1.00      1.00         4

       accuracy                           1.00        20
      macro avg       1.00      1.00      1.00        20
   weighted avg       1.00      1.00      1.00        20



In [56]:
# ---------------------------------------------------------------------------
# INFERENCIA EN TIEMPO REAL - Pacientes nuevos en ambulancia
# ---------------------------------------------------------------------------

def predecir_especialidad(signos):
    df_p = pd.DataFrame([signos])
    X_n  = scaler.transform(df_p[COLUMNAS_NUMERICAS])
    X_t  = tfidf.transform(df_p['Descripcion_Sintomas']).toarray()
    X    = np.hstack([X_n, X_t])
    idx  = clf.predict(X)[0]
    prob = clf.predict_proba(X)[0]
    return {
        'especialidad': codificador.inverse_transform([idx])[0],
        'confianza'   : float(prob[idx]),
        'ranking'     : sorted(zip(codificador.classes_, prob), key=lambda x: x[1], reverse=True),
    }


pacientes = [
    {'_t':'Hombre 65 anos - Dolor toracico',
     'Edad':65,'Frecuencia_Cardiaca':118,'Presion_Arterial_Sistolica':185,
     'Presion_Arterial_Diastolica':105,'Frecuencia_Respiratoria':22,'Saturacion_O2':95,'Glasgow':15,
     'Descripcion_Sintomas':'dolor precordial opresivo diaforesis brazo izquierdo'},
    {'_t':'Mujer 28 anos - Crisis convulsiva',
     'Edad':28,'Frecuencia_Cardiaca':88,'Presion_Arterial_Sistolica':140,
     'Presion_Arterial_Diastolica':85,'Frecuencia_Respiratoria':18,'Saturacion_O2':96,'Glasgow':7,
     'Descripcion_Sintomas':'convulsiones tonico-clonicas perdida conciencia mordedura lengua'},
    {'_t':'Hombre 45 anos - Disnea severa',
     'Edad':45,'Frecuencia_Cardiaca':125,'Presion_Arterial_Sistolica':120,
     'Presion_Arterial_Diastolica':80,'Frecuencia_Respiratoria':38,'Saturacion_O2':78,'Glasgow':14,
     'Descripcion_Sintomas':'disnea severa sibilancias cianosis crisis asmatica grave'},
    {'_t':'Adolescente 17 anos - Trauma',
     'Edad':17,'Frecuencia_Cardiaca':138,'Presion_Arterial_Sistolica':90,
     'Presion_Arterial_Diastolica':55,'Frecuencia_Respiratoria':28,'Saturacion_O2':92,'Glasgow':12,
     'Descripcion_Sintomas':'fractura expuesta femur sangrado abundante accidente moto'},
    {'_t':'Mujer 55 anos - Abdomen agudo',
     'Edad':55,'Frecuencia_Cardiaca':105,'Presion_Arterial_Sistolica':110,
     'Presion_Arterial_Diastolica':70,'Frecuencia_Respiratoria':24,'Saturacion_O2':97,'Glasgow':15,
     'Descripcion_Sintomas':'dolor abdominal difuso defensa muscular rebote positivo'},
]

print('INFERENCIA EN TIEMPO REAL - SISTEMA DE APOYO PREHOSPITALARIO')
print('=' * 70)
for p in pacientes:
    titulo = p.pop('_t')
    res    = predecir_especialidad(p)
    print(f'\nPaciente     : {titulo}')
    print(f'  FC = {p["Frecuencia_Cardiaca"]} lpm | SpO2 = {p["Saturacion_O2"]} % | '
          f'GCS = {p["Glasgow"]} | FR = {p["Frecuencia_Respiratoria"]} rpm')
    print(f'  Sintomas     : {p["Descripcion_Sintomas"][:68]}')
    print(f'  Especialidad : {res["especialidad"].upper()}  (Confianza: {res["confianza"]:.1%})')
    print(f'  Ranking top3 : ' + ' | '.join(f'{c}: {prob:.0%}' for c,prob in res['ranking'][:3]))


INFERENCIA EN TIEMPO REAL - SISTEMA DE APOYO PREHOSPITALARIO

Paciente     : Hombre 65 anos - Dolor toracico
  FC = 118 lpm | SpO2 = 95 % | GCS = 15 | FR = 22 rpm
  Sintomas     : dolor precordial opresivo diaforesis brazo izquierdo
  Especialidad : CARDIOLOGIA  (Confianza: 95.7%)
  Ranking top3 : Cardiologia: 96% | Cirugia_General: 3% | Neurologia: 1%

Paciente     : Mujer 28 anos - Crisis convulsiva
  FC = 88 lpm | SpO2 = 96 % | GCS = 7 | FR = 18 rpm
  Sintomas     : convulsiones tonico-clonicas perdida conciencia mordedura lengua
  Especialidad : NEUROLOGIA  (Confianza: 91.2%)
  Ranking top3 : Neurologia: 91% | Traumatologia: 8% | Cardiologia: 1%

Paciente     : Hombre 45 anos - Disnea severa
  FC = 125 lpm | SpO2 = 78 % | GCS = 14 | FR = 38 rpm
  Sintomas     : disnea severa sibilancias cianosis crisis asmatica grave
  Especialidad : NEUMOLOGIA  (Confianza: 99.3%)
  Ranking top3 : Neumologia: 99% | Traumatologia: 0% | Cirugia_General: 0%

Paciente     : Adolescente 17 anos - Trauma